# Notebook 4: Create Prescription table
Creates 30 prescriptions linked to diagnoses and providers with full coverage rules.

In [ ]:
import random
from pyspark.sql import functions as F

SEED = 20260404
random.seed(SEED)

diag_rows = spark.table('diagnosis').select('diagnosisId', 'providerId').orderBy('diagnosisId').collect()
providers = [r.providerId for r in spark.table('provider').select('providerId').collect()]

if len(diag_rows) != 20:
    raise ValueError('Expected 20 diagnoses from Notebook 3.')
if len(providers) != 20:
    raise ValueError('Expected 20 providers from Notebook 1.')

In [ ]:
medications = [
    'Lisinopril', 'Metformin', 'Atorvastatin', 'Albuterol', 'Levothyroxine',
    'Omeprazole', 'Sertraline', 'Amlodipine', 'Losartan', 'Gabapentin',
    'Amoxicillin', 'Azithromycin', 'Hydrochlorothiazide', 'Montelukast', 'Fluticasone',
    'Ibuprofen', 'Acetaminophen', 'Prednisone', 'Escitalopram', 'Rosuvastatin'
]
dosages = ['5 mg', '10 mg', '20 mg', '25 mg', '50 mg', '100 mg']
frequencies = ['Once daily', 'Twice daily', 'Every 8 hours', 'At bedtime', 'As needed']

prescriptions = []

# First 20 rows guarantee every diagnosis is treated at least once and every provider appears at least once.
for i, d in enumerate(diag_rows, start=1):
    prescriptions.append({
        'rxNumber': f"RX{i:04d}",
        'diagnosisId': d.diagnosisId,
        'providerId': d.providerId,
        'medication': medications[(i - 1) % len(medications)],
        'dosage': random.choice(dosages),
        'frequency': random.choice(frequencies),
        'refillsRemaining': random.randint(0, 3)
    })

# Add 10 more rows with semi-random uneven distribution over diagnoses/providers.
for i in range(21, 31):
    d = random.choice(diag_rows)
    prescriptions.append({
        'rxNumber': f"RX{i:04d}",
        'diagnosisId': d.diagnosisId,
        'providerId': d.providerId,
        'medication': random.choice(medications),
        'dosage': random.choice(dosages),
        'frequency': random.choice(frequencies),
        'refillsRemaining': random.randint(0, 3)
    })

prescription_df = spark.createDataFrame(prescriptions)
prescription_df = prescription_df.select(
    'rxNumber', 'diagnosisId', 'providerId', 'medication', 'dosage', 'frequency', 'refillsRemaining'
)
prescription_df = prescription_df.withColumn('refillsRemaining', F.col('refillsRemaining').cast('int'))
prescription_df.write.mode('overwrite').format('delta').saveAsTable('prescription')

display(spark.table('prescription').orderBy('rxNumber'))

In [ ]:
rx = spark.table('prescription')
diagnosis_coverage = rx.select('diagnosisId').distinct().count()
provider_coverage = rx.select('providerId').distinct().count()
diag_fk_orphans = rx.alias('r').join(spark.table('diagnosis').alias('d'), on='diagnosisId', how='left_anti').count()
provider_fk_orphans = rx.alias('r').join(spark.table('provider').alias('p'), on='providerId', how='left_anti').count()

print('prescription_count_30:', rx.count() == 30)
print('all_20_diagnoses_treated:', diagnosis_coverage == 20)
print('all_20_providers_prescribe:', provider_coverage == 20)
print('diagnosis_fk_valid:', diag_fk_orphans == 0)
print('provider_fk_valid:', provider_fk_orphans == 0)